# Phase 1 (HD) — Data Preparation (Reproducible)

Goal: build one clean, review-level master dataset (`final_dataset.csv`) where **one row = one review**.

Data sources (CSV):
- `archive/reviews.csv`
- `archive/sentiment_labels.csv`
- `archive/products.csv`
- `archive/sellers.csv`

Merge order:
1) `reviews` + `sentiment_labels` on `review_id`
2) + `products` on `product_id`
3) + `sellers` on `seller_id` (from products)

Notes:
- `fake_confidence_score` is kept in the dataset for traceability, but treat it as potential target leakage in modeling unless you justify it.
- `monthly_trends.csv` is aggregated by month and is not used for review-level prediction.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

here = Path.cwd()
candidates = [here, here.parent]
root = None
for c in candidates:
    if (c / "archive").exists() and (c / "phase1").exists():
        root = c
        break
if root is None:
    root = here

archive_dir = root / "archive"
out_dir = root / "phase1"
out_dir.mkdir(parents=True, exist_ok=True)

reviews_path = archive_dir / "reviews.csv"
sent_path = archive_dir / "sentiment_labels.csv"
products_path = archive_dir / "products.csv"
sellers_path = archive_dir / "sellers.csv"

for p in [reviews_path, sent_path, products_path, sellers_path]:
    if not p.exists():
        raise FileNotFoundError(f"Missing required file: {p}")

print("Project root:", root)
print("Archive dir:", archive_dir)
print("Output dir:", out_dir)

## Step 1 — Load datasets

In [ ]:
reviews = pd.read_csv(reviews_path)
sent = pd.read_csv(sent_path)
products = pd.read_csv(products_path)
sellers = pd.read_csv(sellers_path)

print("reviews:", reviews.shape)
print("sentiment_labels:", sent.shape)
print("products:", products.shape)
print("sellers:", sellers.shape)

display(reviews.head(2))
display(sent.head(2))

## Step 2 — Merge datasets (core)

- `reviews` + `sentiment_labels` on `review_id`
- then merge `products` on `product_id`
- then merge `sellers` on `seller_id`

In [ ]:
df = reviews.merge(sent, on="review_id", how="inner", suffixes=("", "_sent"))
df = df.rename(
    columns={
        "star_rating_sent": "sent_star_rating",
        "is_fake_review_sent": "sent_is_fake_review",
    }
)

df = df.merge(products, on="product_id", how="left", suffixes=("", "_product"))

df = df.rename(
    columns={
        "num_images": "product_num_images",
        "total_reviews": "product_total_reviews",
        "avg_rating": "product_avg_rating",
    }
)

df = df.merge(sellers, on="seller_id", how="left", suffixes=("", "_seller"))

if "total_reviews_seller" in df.columns:
    df = df.rename(columns={"total_reviews_seller": "seller_total_reviews"})

print("After merge:", df.shape)
df.head(2)

## Step 3 — Data cleaning & checks

Checks required by Phase 1:
- duplicate `review_id`
- missing values overview
- label validity (`is_fake_review` in {0,1})
- label consistency between sources (`is_fake_review` vs `sent_is_fake_review`)

In [ ]:
before = len(df)
df = df.drop_duplicates(subset=["review_id"], keep="first")
print("Removed duplicate review_id:", before - len(df))

df["is_fake_review"] = df["is_fake_review"].astype(str).str.strip()
valid_mask = df["is_fake_review"].isin(["0", "1"])
invalid_count = int((~valid_mask).sum())
df = df.loc[valid_mask].copy()
print("Removed invalid/missing labels:", invalid_count)

if "sent_is_fake_review" in df.columns:
    mismatch = (df["sent_is_fake_review"].astype(str).str.strip() != df["is_fake_review"]).sum()
    print("Label mismatches (reviews vs sentiment_labels):", int(mismatch))
else:
    print("sent_is_fake_review not found")

print("Class balance:")
display(df["is_fake_review"].value_counts())
display(df["is_fake_review"].value_counts(normalize=True))

missing = df.isna().sum().sort_values(ascending=False)
display(missing.head(20))

## Step 4 — Save final dataset

Outputs:
- `phase1/final_dataset.csv`

Optional outputs (set flags in the code cell below):
- `phase1/phase1_data_audit.md`
- `phase1/phase1_requirements_check.md`

In [ ]:
WRITE_AUDIT_MD = False
WRITE_REQUIREMENTS_MD = False

final_path = out_dir / "final_dataset.csv"
df.to_csv(final_path, index=False)
print("Wrote:", final_path)

audit_path = out_dir / "phase1_data_audit.md"
req_path = out_dir / "phase1_requirements_check.md"

rows = len(df)
uniq = df["review_id"].nunique()
dup = rows - uniq
mismatch = int((df.get("sent_is_fake_review", pd.Series(["0"] * rows)).astype(str).str.strip() != df["is_fake_review"]).sum()) if "sent_is_fake_review" in df.columns else 0
class_counts = df["is_fake_review"].value_counts().sort_index()

missing_top = df.isna().sum().sort_values(ascending=False).head(20)

audit_lines = []
audit_lines.append("# Phase 1 Data Audit")
audit_lines.append("")
audit_lines.append("## Inputs")
audit_lines.append(f"- reviews.csv: {len(reviews)} rows")
audit_lines.append(f"- sentiment_labels.csv: {len(sent)} rows")
audit_lines.append(f"- products.csv: {len(products)} rows")
audit_lines.append(f"- sellers.csv: {len(sellers)} rows")
audit_lines.append("")
audit_lines.append("## Merge Results")
audit_lines.append(f"- Rows: {rows}")
audit_lines.append(f"- Unique review_id: {uniq}")
audit_lines.append(f"- Duplicate review_id: {dup}")
audit_lines.append(f"- Label mismatches (reviews vs sentiment_labels): {mismatch}")
audit_lines.append("")
audit_lines.append("## Class Balance (is_fake_review)")
for k, v in class_counts.items():
    pct = round(100 * v / rows, 2) if rows else 0
    audit_lines.append(f"- {k}: {int(v)} ({pct}%)")
audit_lines.append("")
audit_lines.append("## Missing Values (Top 20 Columns)")
audit_lines.append("| column | missing |")
audit_lines.append("|---|---:|")
for col, m in missing_top.items():
    audit_lines.append(f"| {col} | {int(m)} |")
audit_lines.append("")
audit_lines.append("## Leakage Note")
audit_lines.append("- fake_confidence_score exists: " + ("YES" if "fake_confidence_score" in df.columns else "NO"))

if WRITE_AUDIT_MD:
    audit_path.write_text("\n".join(audit_lines), encoding="utf-8")
    print("Wrote:", audit_path)

req_lines = []
req_lines.append("# Phase 1 Requirements Check (final_dataset.csv)")
req_lines.append("")
req_lines.append("## Summary")
req_lines.append(f"- Rows: {rows}")
req_lines.append(f"- Unique review_id: {uniq}")
req_lines.append(f"- Duplicate review_id: {dup}")
req_lines.append(f"- Label mismatches (is_fake_review vs sent_is_fake_review): {mismatch}")
req_lines.append("")
req_lines.append("## Requirement-by-Requirement")
req_lines.append("- One row = one review: " + ("PASS" if dup == 0 else "FAIL"))
req_lines.append("- Target label is_fake_review exists: " + ("PASS" if "is_fake_review" in df.columns else "FAIL"))
req_lines.append("- Merged linguistic features exist: " + ("PASS" if "sentiment_label" in df.columns else "FAIL"))
req_lines.append("- Product context exists: " + ("PASS" if "true_quality" in df.columns else "FAIL"))
req_lines.append("- Seller context exists: " + ("PASS" if "fake_rate" in df.columns else "FAIL"))
req_lines.append("- fake_confidence_score leakage note: " + ("PRESENT" if "fake_confidence_score" in df.columns else "ABSENT"))

if WRITE_REQUIREMENTS_MD:
    req_path.write_text("\n".join(req_lines), encoding="utf-8")
    print("Wrote:", req_path)